## 0. Prérequis

1. **Exécution → Modifier le type d'exécution → T4 GPU**.
2. Vérifier que le GPU est bien alloué (cellule suivante).
3. Compter **moins de 10 minutes** par entraînement.

> Toute expérience part de `src/config.py` et ne modifie **qu'un seul**
> hyperparamètre par rapport à la baseline.


In [ ]:
!nvidia-smi

## 1. Cloner le dépôt et installer les dépendances

In [ ]:
import os

REPO_URL = "https://github.com/jaouheraya12-create/transformer-hyperparam-study.git"
REPO_DIR = "transformer-hyperparam-study"

# Idempotent : on clone seulement si le dossier n'existe pas déjà.
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

%cd {REPO_DIR}
!git pull --ff-only


In [ ]:
!pip install -q -r requirements.txt

## 2. Vérifier la configuration

On vérifie que le paquet `src` est importable et que la configuration de
référence est bien celle validée par l'équipe.


In [ ]:
!python -c "import src.config; print('config OK')"

from src.config import Config

cfg = Config()
print(cfg.resume())
print("Splits :", cfg.train_size, "/", cfg.val_size, "/", cfg.test_size)


# 03 — Batch size, optimizer, temps et mémoire GPU (Aya)

**Expériences 9 à 12.** Objectif : mesurer le compromis **qualité / temps /
mémoire GPU**.

| Exp. | Nom | `batch_size` | `optimizer_type` | `learning_rate` |
|---|---|---|---|---|
| 9 | `batch_16` | 16 | adamw | `2e-5` |
| 10 | `batch_64` | 64 | adamw | `2e-5` |
| 11 | `sgd_lr_2e-5` | 32 | sgd | `2e-5` |
| 12 | `sgd_lr_1e-2` | 32 | sgd | `1e-2` |

> L'expérience 12 change **deux** champs (optimiseur + LR) : c'est volontaire,
> car SGD avec `2e-5` ne converge pas. À justifier dans `observations`.


## Expérience 9 — batch 16

In [ ]:
!python -m src.train --id 9 --owner Aya --name batch_16 --batch-size 16


## Expérience 10 — batch 64

In [ ]:
!python -m src.train --id 10 --owner Aya --name batch_64 --batch-size 64


## Expérience 11 — SGD, lr 2e-5 (référence de comparaison)

In [ ]:
!python -m src.train --id 11 --owner Aya --name sgd_lr_2e-5 --optimizer sgd


## Expérience 12 — SGD, lr 1e-2 (LR adapté)

In [ ]:
!python -m src.train --id 12 --owner Aya --name sgd_lr_1e-2 --optimizer sgd --lr 1e-2


## Tableau temps / mémoire GPU

In [ ]:
df = pd.read_csv("results/all_experiments.csv")
mine = df[(df["owner"] == "Aya") & (df["id"].isin([9, 10, 11, 12]))]

colonnes = ["id", "name", "batch_size", "optimizer", "learning_rate",
            "train_time_s", "gpu_mem_alloc_mb", "gpu_mem_reserved_mb", "f1"]
mine[colonnes].sort_values("id")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(mine["name"], mine["train_time_s"])
axes[0].set_title("Temps d'entraînement (s)")
axes[0].tick_params(axis="x", rotation=20)

axes[1].scatter(mine["train_time_s"], mine["f1"])
for _, ligne in mine.iterrows():
    axes[1].annotate(ligne["name"], (ligne["train_time_s"], ligne["f1"]), fontsize=8)
axes[1].set_xlabel("Temps (s)")
axes[1].set_ylabel("F1 (test)")
axes[1].set_title("Compromis temps / qualité")

plt.tight_layout()
plt.savefig("results/aya/figures/temps_memoire_f1.png", dpi=150)
plt.show()


## Analyse mémoire GPU

`metrics.json` contient `gpu_mem_alloc_mb` (pic alloué par PyTorch) et
`gpu_mem_reserved_mb` (pic réservé par le cache CUDA). Commenter l'effet du
batch size (mémoire ~linéaire en batch) et de l'optimiseur (AdamW stocke deux
moments par paramètre, donc plus de mémoire que SGD).
